In [1]:
!pwd

/home/hgkahng/Workspaces/soft-prompt/notebooks/subj


In [2]:
import os
import sys
sys.path.insert(
    0, os.path.abspath('../../')
)

import json
import yaml

from pathlib import Path
from rich.console import Console
from rich.table import Table

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
root_dir = Path("../../").resolve()
print("Root directory:", root_dir)

Root directory: /home/hgkahng/Workspaces/soft-prompt


## 1. Load Oracle Data

In [4]:
from typing import Union, Tuple

def load_oracle_subj_data(directory: Union[str, Path]) -> Tuple[np.ndarray]:
    
    _directory = Path(directory).resolve()
    X_train = np.load(_directory / "train.features.npy")
    y_train = np.load(_directory / "train.labels.npy")
    X_test = np.load(_directory / "test.features.npy")
    y_test = np.load(_directory / "test.labels.npy")
    
    return (X_train, y_train, X_test, y_test)


emb_save_dir = root_dir / "data/subj/embeddings/openai/text-embedding-3-small"
X_train, y_train, X_test, y_test = load_oracle_subj_data(emb_save_dir)

In [5]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((8000, 1536), (8000,), (2000, 1536), (2000,))

## 2. Load Synthetic Data

In [ ]:
# TODO: Deprecated

load_dir = root_dir / "results/subj/2025-05-09_20:04:58"  # hard
load_dir = root_dir / "results/subj/2025-05-09_19:40:23"  # soft
load_dir = root_dir / "results/subj/2025-05-09_20:51:10"  # hard + CoT
load_dir = root_dir / "results/subj/2025-05-09_20:47:29"  # soft + CoT

# Prompt: write 'movie-related' text + system prompt has movie related instructions
load_dir = root_dir / "results/subj/2025-05-09_21:45:33"  # soft
load_dir = root_dir / "results/subj/2025-05-09_21:45:44"  # hard
load_dir = root_dir / "results/subj/2025-05-09_22:14:33"  # hard + CoT
load_dir = root_dir / "results/subj/2025-05-09_22:09:45"  # soft + CoT

# Prompt: write 'movie-related' text sample
load_dir = root_dir / "results/subj/2025-05-09_23:11:32"  # soft + CoT
load_dir = root_dir / "results/subj/2025-05-09_23:56:44"  # hard + CoT

In [23]:
load_dir = root_dir / "results/subj/gemini-2.0-flash/hard"
load_dir = root_dir / "results/subj/gemini-2.0-flash(most_simple)/soft+cot"
load_dir = root_dir / "results/subj/gemini-2.0-flash(long)/hard+cot"

print(*os.listdir(load_dir), sep="\n")

template.jsonl
embeddings
config.yaml
data.jsonl
template_formatted.txt


Print configurations

In [24]:
# Print configurations
with open(load_dir / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

table = Table(title="Configuration(s)")
table.add_column("Name", justify="right", style="white", no_wrap=True)
table.add_column("Value", justify="left", style='red' if cfg['hard'] else 'cyan')
_ = [table.add_row(k, str(v)) for k, v in cfg.items()]

console = Console()
console.print(table)

         Configuration(s)          
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃         Name ┃ Value            ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│   batch_size │ 50               │
│          cot │ True             │
│         data │ subj             │
│         hard │ True             │
│ log_interval │ 2                │
│   max_tokens │ 16384            │
│        model │ gemini-2.0-flash │
│ num_examples │ None             │
│  sample_size │ 20000            │
│  temperature │ 1.0              │
└──────────────┴──────────────────┘

In [25]:
with open(load_dir / 'template_formatted.txt') as f:
    template_example = "".join(f.readlines())

print(template_example)

system:
You are tasked with generating realistic text to train a subjectivity classifier.
Use a subjectivity scale where 0 represents an objective statement (e.g., factual descriptions, neutral narration) and1 represents a subjective statement (e.g., personal opinions, emotional expressions, or evaluative language.)
Use everyday language as if written on some online platform.
Text that you write should generally be between 10 and 50 words.
Before writing your final answer, please think step-by-step.
Ensure that you MUST provide your step-by-step thinking process in the final output.

human:
Generate a text sample with a subjectivity score of 1.



In [26]:
with open(load_dir / "data.jsonl", "r") as f:
    data = [json.loads(line) for line in f]

print(f"Data size: {len(data):,}")

Data size: 20,048


In [27]:
data[0]

{'label': 1.0,
 'text': 'That movie was absolutely terrible! What a waste of my time and money. ',
 'reasoning': 'The user requested a text sample with a subjectivity score of 1, meaning it should be highly subjective. I will generate a short sentence expressing a strong personal opinion about a movie.'}

In [28]:
labels = [d['label'] for d in data]
labels = np.array(labels)

hard_labels = (labels > 0.5).astype(int)
soft_labels = labels.copy()

print("Hard labels, shape:", hard_labels.shape)
print("Soft labels, shape:", soft_labels.shape)

Hard labels, shape: (20048,)
Soft labels, shape: (20048,)


In [29]:
embeddings = np.load(
    load_dir / "embeddings/openai/text-embedding-3-small/data.npy"
)
print(embeddings.shape)

(20048, 1536)


In [30]:
assert len(hard_labels) == embeddings.shape[0]
assert len(soft_labels) == embeddings.shape[0]

In [31]:
# aliasing for readabillity
X_syn = embeddings.copy()
y_syn_hard = hard_labels.copy()
y_syn_soft = soft_labels.copy()

In [32]:
pd.Series(y_syn_hard).value_counts().to_frame()

,count
0,10039
1,10009


In [33]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## Downstream Classification Performance

In [34]:
from softprompt.algorithms.helpers import evaluate_logreg_cv_experiment

`Standard Logistic Regression`

In [35]:
len(X_train), len(X_test), len(X_syn)

(8000, 2000, 20048)

In [36]:
N = len(X_train) # Changed from X_train to X_syn
ratios = (0.01, 0.05, 0.10, 0.25, 0.50, 1.00)
subsample_sizes = [int(N * r) for r in ratios]

# Select which model variant to run in this loop
# Options: 'soft' or 'standard'
current_model_variant = 'standard'

for i, subsample_size in enumerate(subsample_sizes):

    from rich.console import Console
    console = Console()
    console.print(
        f">> Model Variant: [bold yellow]{current_model_variant.upper()}[/bold yellow], "
        f"Subsample Ratio: {ratios[i]:.2f} (Size: {subsample_size:,})"
    )

    # (optional) Convert y_test probabilities to hard labels for y_test_hard argument
    y_test_hard = (y_test > 0.5).astype(int)
    tr_metrics, te_metrics = evaluate_logreg_cv_experiment(
        X_train_full=X_syn,
        y_train_full_probs=y_syn_soft,
        X_test=X_test,
        y_test_hard=y_test_hard,
        subsample_size=subsample_size,
        model_variant=current_model_variant, 
        bootstrap=True,
        n_trials=5,
    )

    from rich.table import Table
    table = Table(title=f"Metrics (r={ratios[i]:.2f})")
    table.add_column("Metric Name", style='cyan', no_wrap=True, justify='left')
    table.add_column("Train", style='blue', justify='center')
    table.add_column("Test", style='white', justify='center')
    for name in tr_metrics.keys():  # assuming that keys are identical
        tr_mean_, tr_std_ = tr_metrics[name]
        tr_metric_value_str = f"{tr_mean_:.4f} (± {tr_std_:.4f})"
        te_mean_, te_std_ = te_metrics[name]
        te_metric_value_str = f"{te_mean_:.4f} (± {te_std_:.4f})"
        table.add_row(
            name,
            tr_metric_value_str,
            te_metric_value_str,
        )
    console.print(table);

    console.print("-" * 50)

>> Model Variant: STANDARD, Subsample Ratio: 0.01 (Size: 80)

                   Metrics (r=0.01)                    
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Metric Name ┃       Train       ┃       Test        ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ accuracy    │ 1.0000 (± 0.0000) │ 0.7348 (± 0.0273) │
│ precision   │ 1.0000 (± 0.0000) │ 0.8440 (± 0.0262) │
│ recall      │ 1.0000 (± 0.0001) │ 0.5566 (± 0.0842) │
│ f1_score    │ 1.0000 (± 0.0000) │ 0.6669 (± 0.0561) │
│ roc_auc     │ 1.0000 (± 0.0000) │ 0.8202 (± 0.0202) │
│ auprc       │ 1.0000 (± 0.0000) │ 0.8292 (± 0.0204) │
│ ece         │ 0.0037 (± 0.0009) │ 0.0879 (± 0.0258) │
└─────────────┴───────────────────┴───────────────────┘

--------------------------------------------------

>> Model Variant: STANDARD, Subsample Ratio: 0.05 (Size: 400)

                   Metrics (r=0.05)                    
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Metric Name ┃       Train       ┃       Test        ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ accuracy    │ 1.0000 (± 0.0000) │ 0.7112 (± 0.0196) │
│ precision   │ 1.0000 (± 0.0000) │ 0.9060 (± 0.0282) │
│ recall      │ 1.0000 (± 0.0000) │ 0.4515 (± 0.0619) │
│ f1_score    │ 1.0000 (± 0.0000) │ 0.5998 (± 0.0473) │
│ roc_auc     │ 1.0000 (± 0.0000) │ 0.8310 (± 0.0104) │
│ auprc       │ 1.0000 (± 0.0000) │ 0.8416 (± 0.0102) │
│ ece         │ 0.0024 (± 0.0004) │ 0.1389 (± 0.0270) │
└─────────────┴───────────────────┴───────────────────┘

...

                   Metrics (r=0.50)                    
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Metric Name ┃       Train       ┃       Test        ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ accuracy    │ 1.0000 (± 0.0000) │ 0.7031 (± 0.0051) │
│ precision   │ 1.0000 (± 0.0000) │ 0.9148 (± 0.0164) │
│ recall      │ 1.0000 (± 0.0000) │ 0.4261 (± 0.0208) │
│ f1_score    │ 1.0000 (± 0.0000) │ 0.5809 (± 0.0160) │
│ roc_auc     │ 1.0000 (± 0.0000) │ 0.8362 (± 0.0025) │
│ auprc       │ 1.0000 (± 0.0000) │ 0.8461 (± 0.0027) │
│ ece         │ 0.0014 (± 0.0000) │ 0.1549 (± 0.0141) │
└─────────────┴───────────────────┴───────────────────┘

--------------------------------------------------

>> Model Variant: STANDARD, Subsample Ratio: 1.00 (Size: 8,000)

                   Metrics (r=1.00)                    
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Metric Name ┃       Train       ┃       Test        ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ accuracy    │ 1.0000 (± 0.0000) │ 0.6949 (± 0.0058) │
│ precision   │ 1.0000 (± 0.0000) │ 0.9275 (± 0.0086) │
│ recall      │ 1.0000 (± 0.0000) │ 0.4004 (± 0.0163) │
│ f1_score    │ 1.0000 (± 0.0000) │ 0.5591 (± 0.0147) │
│ roc_auc     │ 1.0000 (± 0.0000) │ 0.8344 (± 0.0017) │
│ auprc       │ 1.0000 (± 0.0000) │ 0.8441 (± 0.0019) │
│ ece         │ 0.0007 (± 0.0000) │ 0.1795 (± 0.0094) │
└─────────────┴───────────────────┴───────────────────┘

--------------------------------------------------

## Margin-based Filtering

In [37]:
console = Console()

margin_to_metrics = {}
margins = (0.05, 0.10, 0.15, 0.20, 0.25, 0.30)

# Select which model variant to run in this loop
# Options: 'soft' or 'standard' or 'gce'
current_model_variant = 'standard'

for _, m in enumerate(margins):
    console.rule(f"[bold cyan]Margin: {m:.2f}[/bold cyan]")

    # Filter data based on margin using y_syn_soft (1D probabilities)
    margin_mask = \
        (pd.Series(y_syn_soft) <= (0.5 - m)) | \
        (pd.Series(y_syn_soft) >= (0.5 + m))
    effective_indices = np.where(margin_mask.values)[0]

    if len(effective_indices) == 0:
        console.print(f">> Effective pool size is 0 with margin = {m}. Skipping.")
        margin_to_metrics[m] = {'train': {}, 'test': {}} # Store empty results
        continue

    X_syn_filtered = X_syn[effective_indices]
    y_syn_soft_filtered = y_syn_soft[effective_indices] # This is 1D
    N_filtered = len(X_syn_filtered)
    console.print(f">> Effective pool size is {N_filtered:,} with margin = {m}")

    # Store metrics for this margin across different subsample ratios
    from typing import Dict, List, Tuple
    train_metrics_for_margin_all_ratios: Dict[str, List[List[float]]] = {}
    test_metrics_for_margin_all_ratios: Dict[str, List[List[float]]] = {}
    
    N = len(X_train)  # 
    ratios = (0.01, 0.05, 0.10, 0.25, 0.50, 1.00)
    subsample_sizes = [int(N * r) for r in ratios]

    for i, subsample_size in enumerate(subsample_sizes):
        
        console.print(f"\t>> Model: [bold yellow]{current_model_variant.upper()}[/bold yellow], "
                      f"Subsample Ratio: {ratios[i]:.2f} (Size: {subsample_size:,})")

        # Convert y_test probabilities to hard labels for y_test_hard argument
        y_test_hard_labels = (y_test > 0.5).astype(int)

        tr_metrics, te_metrics = evaluate_logreg_cv_experiment(
            X_train_full=X_syn_filtered,
            y_train_full_probs=y_syn_soft_filtered, # Pass the filtered 1D soft labels
            X_test=X_test,
            y_test_hard=y_test_hard_labels,
            subsample_size=subsample_size,
            model_variant=current_model_variant,
            bootstrap=True,
            n_trials=50, # Fewer trials for faster example run
        )
        
        # Store all metrics for this ratio
        for metric_name, (mean_val, std_val) in tr_metrics.items():
            if metric_name not in train_metrics_for_margin_all_ratios:
                train_metrics_for_margin_all_ratios[metric_name] = []
            train_metrics_for_margin_all_ratios[metric_name].append([mean_val, std_val])
        
        for metric_name, (mean_val, std_val) in te_metrics.items():
            if metric_name not in test_metrics_for_margin_all_ratios:
                test_metrics_for_margin_all_ratios[metric_name] = []
            test_metrics_for_margin_all_ratios[metric_name].append([mean_val, std_val])

        from rich.table import Table
        table = Table(title=f"Metrics (m={m:.2f}, r={ratios[i]:.2f})")
        table.add_column("Metric Name", style='cyan', no_wrap=True, justify='left')
        table.add_column("Train", style='blue', justify='center')
        table.add_column("Test", style='white', justify='center')
        for name in tr_metrics.keys():
            tr_mean_, tr_std_ = tr_metrics[name]
            tr_metric_value_str = f"{tr_mean_:.4f} (± {tr_std_:.4f})"
            te_mean_, te_std_ = te_metrics[name]
            te_metric_value_str = f"{te_mean_:.4f} (± {te_std_:.4f})"
            table.add_row(
                name,
                tr_metric_value_str,
                te_metric_value_str,
            )
        console.print(table);

    margin_to_metrics[m] = {
        'train': train_metrics_for_margin_all_ratios,
        'test': test_metrics_for_margin_all_ratios,
    }

    console.print("-" * 50)

────────────────────────────────────────────────── Margin: 0.05 ───────────────────────────────────────────────────

>> Effective pool size is 18,087 with margin = 0.05

>> Model: STANDARD, Subsample Ratio: 0.01 (Size: 80)

               Metrics (m=0.05, r=0.01)                
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Metric Name ┃       Train       ┃       Test        ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ accuracy    │ 0.9148 (± 0.0101) │ 0.6599 (± 0.0965) │
│ precision   │ 0.9360 (± 0.0306) │ 0.6051 (± 0.0840) │
│ recall      │ 0.8949 (± 0.0328) │ 0.9570 (± 0.0393) │
│ f1_score    │ 0.9140 (± 0.0103) │ 0.7362 (± 0.0509) │
│ roc_auc     │ 0.9729 (± 0.0087) │ 0.8910 (± 0.0306) │
│ auprc       │ 0.9782 (± 0.0057) │ 0.8956 (± 0.0313) │
│ ece         │ 0.0464 (± 0.0213) │ 0.2282 (± 0.0724) │
└─────────────┴───────────────────┴───────────────────┘

>> Model: STANDARD, Subsample Ratio: 0.05 (Size: 400)

...

>> Model: STANDARD, Subsample Ratio: 0.50 (Size: 4,000)

               Metrics (m=0.30, r=0.50)                
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Metric Name ┃       Train       ┃       Test        ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ accuracy    │ 0.9997 (± 0.0001) │ 0.8004 (± 0.0106) │
│ precision   │ 0.9998 (± 0.0002) │ 0.7409 (± 0.0164) │
│ recall      │ 0.9996 (± 0.0002) │ 0.9044 (± 0.0099) │
│ f1_score    │ 0.9997 (± 0.0001) │ 0.8143 (± 0.0068) │
│ roc_auc     │ 1.0000 (± 0.0000) │ 0.9090 (± 0.0034) │
│ auprc       │ 1.0000 (± 0.0000) │ 0.9174 (± 0.0033) │
│ ece         │ 0.0024 (± 0.0002) │ 0.1163 (± 0.0134) │
└─────────────┴───────────────────┴───────────────────┘

>> Model: STANDARD, Subsample Ratio: 1.00 (Size: 8,000)

               Metrics (m=0.30, r=1.00)                
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Metric Name ┃       Train       ┃       Test        ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ accuracy    │ 0.9998 (± 0.0001) │ 0.8061 (± 0.0086) │
│ precision   │ 0.9999 (± 0.0001) │ 0.7487 (± 0.0130) │
│ recall      │ 0.9998 (± 0.0002) │ 0.9023 (± 0.0074) │
│ f1_score    │ 0.9998 (± 0.0001) │ 0.8182 (± 0.0058) │
│ roc_auc     │ 1.0000 (± 0.0000) │ 0.9103 (± 0.0027) │
│ auprc       │ 1.0000 (± 0.0000) │ 0.9186 (± 0.0027) │
│ ece         │ 0.0016 (± 0.0001) │ 0.1058 (± 0.0098) │
└─────────────┴───────────────────┴───────────────────┘

--------------------------------------------------